In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
import datetime

pools = {
    "yb_cbBTC": "0x83f24023d15d835a213df24fd309c47dAb5BEb32",
    "yb_wBTC": "0xD9FF8396554A0d18B2CFbeC53e1979b7ecCe8373",
    "yb_tBTC": "0xf1F435B05D255a5dBdE37333C0f61DA6F69c6127",
}
decimals = {
    "yb_cbBTC": 8,
    "yb_wBTC": 8,
    "yb_tBTC": 18,
}

In [ ]:
pool = pools["yb_wBTC"]
# pool = pools['yb_tBTC']
# pool = pools['yb_cbBTC']
# read pandas dataframe, all integers except header
pools_data = {}
donation_duration = 7 * 86_400
protection_period = 60
lp_threshold = 1.0
shares_max_ratio = 0.1
ts_null = 1758733775  # ts post ape_tax bump
# ts_null = 1760147999 # ts post shakeout and vp = xcp_p/2
# ts_null = 1760140800 # ts post shakeout and vp = xcp_p/2
# ts_null = 1760180400 # ts post shakeout and vp = xcp_p/2
# ts_null = 1
for pool, address in pools.items():
    filename = f"data_events/{address}.csv"
    data = pd.read_csv(filename)
    print(len(data))
    idx_null = np.where(data["timestamp"] > ts_null)[0][0]
    data = data[idx_null::]
    ts = np.array(data["timestamp"]).astype(float)
    ts_dt = pd.to_datetime(ts, unit="s")
    blocks = np.array(data["block"]).astype(float)
    virtual_price = np.array(data["virtual_price"]).astype(float) / 1e18
    xcp_profit = np.array(data["xcp_profit"]).astype(float) / 1e18
    price_oracle = np.array(data["price_oracle"]).astype(float) / 1e18
    price_scale = np.array(data["price_scale"]).astype(float) / 1e18
    donation_shares = np.array(data["donation_shares"]).astype(float) / 1e18
    last_donation_release_ts = np.array(data["last_donation_release_ts"]).astype(float)
    donation_protection_expiry_ts = np.array(data["donation_protection_expiry_ts"]).astype(float)
    total_supply = np.array(data["totalSupply"]).astype(float) / 1e18
    D = np.array(data["D"]).astype(float) / 1e18
    balances_0 = np.array(data["balances_0"]).astype(float) / 1e18
    balances_1 = np.array(data["balances_1"]).astype(float) / 10 ** decimals[pool]
    spot_price_in = 1 / (np.array(data["spot_price_in"]).astype(float) / 10 ** decimals[pool])
    spot_price_out = np.array(data["spot_price_out"]).astype(float) / 10**18 / 1e-5
    lp_price = np.array(data["lp_price"]).astype(float) / 1e18
    # calc metrics
    xcp_profit_half = (xcp_profit - 1) / 2 + 1
    protection_factor = np.clip((donation_protection_expiry_ts - ts) / protection_period, 0, 1)
    t_elapsed = ts - last_donation_release_ts
    unlocked_shares = np.clip(donation_shares * t_elapsed / donation_duration, 0, donation_shares)
    vp_xcp_half_gap = virtual_price - xcp_profit_half
    available_shares = unlocked_shares * (1 - protection_factor)
    donations_proportion = donation_shares / total_supply
    unlocked_proportion = unlocked_shares / total_supply
    value_oracle = balances_0 + balances_1 * price_oracle
    value_pscale = balances_0 + balances_1 * price_scale
    virtual_price_growth = virtual_price - virtual_price[0]
    xcp_profit_growth = xcp_profit - xcp_profit[0]
    xcp_profit_half_growth = xcp_profit_half - xcp_profit_half[0]
    pool_balance = balances_0 / (balances_1 * (spot_price_in + spot_price_out) / 2)
    donations_value = donation_shares * lp_price
    available_value = available_shares * lp_price
    spot_mid = (spot_price_in + spot_price_out) / 2
    ps_oracle_diff = abs(price_scale - price_oracle)
    pools_data[pool] = {
        "ts": ts,
        "ts_dt": ts_dt,
        "blocks": blocks,
        "virtual_price": virtual_price,
        "xcp_profit": xcp_profit,
        "price_oracle": price_oracle,
        "price_scale": price_scale,
        "donation_shares": donation_shares,
        "last_donation_release_ts": last_donation_release_ts,
        "donation_protection_expiry_ts": donation_protection_expiry_ts,
        "total_supply": total_supply,
        "D": D,
        "balances_0": balances_0,
        "balances_1": balances_1,
        "spot_price_in": spot_price_in,
        "spot_price_out": spot_price_out,
        "xcp_profit_half": xcp_profit_half,
        "protection_factor": protection_factor,
        "t_elapsed": t_elapsed,
        "unlocked_shares": unlocked_shares,
        "available_shares": available_shares,
        "donations_proportion": donations_proportion,
        "unlocked_proportion": unlocked_proportion,
        "value_oracle": value_oracle,
        "value_pscale": value_pscale,
        "virtual_price_growth": virtual_price_growth,
        "xcp_profit_growth": xcp_profit_growth,
        "xcp_profit_half_growth": xcp_profit_half_growth,
        "pool_balance": pool_balance,
        "lp_price": lp_price,
        "vp_xcp_half_gap": vp_xcp_half_gap,
        "donations_value": donations_value,
        "available_value": available_value,
        "ps_oracle_diff": ps_oracle_diff,
        "spot_mid": spot_mid,
    }

## Price_oracle and price_scale

In [ ]:
pool_key = "yb_tBTC"
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["price_oracle"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["price_scale"])
# rotate x axis labels
plt.xticks(rotation=45)
# legend
plt.legend(["price_oracle", "price_scale", "spot_price_in"])
plt.show()

In [ ]:
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["spot_price_in"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["spot_price_out"])
plt.plot(
    pools_data[pool_key]["ts_dt"],
    (pools_data[pool_key]["spot_price_in"] + pools_data[pool_key]["spot_price_out"]) / 2,
)

# rotate x axis labels
plt.xticks(rotation=45)
# legend
plt.legend(["spot_price_in", "spot_price_out", "spot_price_avg"])
plt.show()

## virtual price and xcp_profit

In [ ]:
# pool_key = "yb_wBTC"
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["xcp_profit"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["xcp_profit_half"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["virtual_price"])
# rotate x axis labels
plt.xticks(rotation=45)

# legend
plt.legend(["xcp_profit", "xcp_profit/2", "virtual_price"])
plt.show()

## Donations

In [ ]:
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["protection_factor"])
plt.xticks(rotation=45)

# legend
plt.legend(["protection_factor"])
plt.show()

In [ ]:
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["donation_shares"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["unlocked_shares"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["available_shares"])
plt.xticks(rotation=45)
plt.legend(["donation_shares", "unlocked_shares", "available_shares"])
plt.show()

In [ ]:
plt.plot(
    pools_data[pool_key]["ts_dt"],
    pools_data[pool_key]["donation_shares"] * pools_data[pool_key]["lp_price"],
)
plt.xticks(rotation=45)

# legend
plt.legend(["donations_value"])
plt.show()

In [ ]:
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["donations_proportion"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["unlocked_proportion"])
plt.xticks(rotation=45)
plt.legend(["donations_proportion", "unlocked_proportion"])
plt.show()

# Compare various pools

In [ ]:
def compare_pools_plots(pools, pools_data, metric):
    fig, ax = plt.subplots(figsize=(10, 4))
    for pool in pools:
        ax.plot(pools_data[pool]["ts_dt"], pools_data[pool][metric], label=pool)
    ax.tick_params(axis="x", rotation=45)
    ax.set_title(metric)
    ax.legend()
    fig.tight_layout()
    return fig, ax


def plot_metrics_panel(pools, pools_data, metrics, t_start=None, t_stop=None):
    if not metrics:
        raise ValueError("metrics list must not be empty")

    fig, axes = plt.subplots(len(metrics), 1, figsize=(6, 3 * len(metrics)), sharex=True)
    if len(metrics) == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        for pool in pools:
            if t_start is None:
                idx_start = 0
            else:
                idx_start = np.where(pools_data[pool]["ts"] > t_start)[0][0]
            if t_stop is None:
                idx_stop = len(pools_data[pool][metric])
            else:
                idx_stop = np.where(pools_data[pool]["ts"] < t_stop)[0][-1]
            ax.plot(
                pools_data[pool]["ts_dt"][idx_start:idx_stop],
                pools_data[pool][metric][idx_start:idx_stop],
                label=pool,
            )
        ax.set_title(metric)
        ax.tick_params(axis="x", rotation=45)
        ax.grid(alpha=0.2)

    axes[-1].set_xlabel("timestamp")
    axes[0].legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.tight_layout()
    return fig, axes


metrics = [
    "protection_factor",
    "virtual_price",
    "xcp_profit",
    "vp_xcp_half_gap",
    "donation_shares",
    "available_shares",
    "donations_value",
    "price_scale",
    "price_oracle",
    "ps_oracle_diff",
    "spot_mid",
    "pool_balance",
]
# metrics = ['virtual_price']
# metrics = ['donation_shares', 'available_shares', 'ps_oracle_diff', 'price_scale','price_oracle']
# metrics = ['donations_value', 'available_value']
# metrics = ['protection_factor']

pools_filtered = {k: v for k, v in pools.items() if k in ["yb_tBTC"]}
t_start = time.time() - 30 * 86_400
t_stop = None
pools_filtered = pools
fig, axes = plot_metrics_panel(pools_filtered, pools_data, metrics, t_start, t_stop)
plt.show()

In [ ]:
metric = "xcp_profit_half"
for pool in pools:
    idx_end = len(pools_data[pool][metric]) - 1
    # t_end = pools_data[pool]['ts'][idx_end]
    t_end = time.time()
    dt = 86_400  # seconds until t_end
    idx_end = np.where(pools_data[pool]["ts"] < t_end)[0][-1]
    for idx_begin in range(0, idx_end):
        t_begin = pools_data[pool]["ts"][idx_begin]
        if t_end - t_begin < dt:
            metric_begin = pools_data[pool][metric][idx_begin]
            metric_end = pools_data[pool][metric][idx_end]
            percent_change = (metric_end - metric_begin) / metric_begin * 100
            hrs_ago = (t_end - t_begin) / 3600
            annualized_growth_rate = (
                (metric_end / metric_begin) ** (365 * 86400 / (t_end - t_begin)) - 1
            ) * 100
            t_begin_utc = datetime.datetime.fromtimestamp(t_begin, datetime.timezone.utc).strftime(
                "%Y/%m/%d %H:%M"
            )
            t_end_utc = datetime.datetime.fromtimestamp(t_end, datetime.timezone.utc).strftime(
                "%Y/%m/%d %H:%M"
            )
            print(f"[{pool}] from {t_begin_utc} to {t_end_utc}")
            print(
                f"pre: {metric_begin:4.5f}, post: {metric_end:4.5f}, diff: {metric_end - metric_begin:4.5f}, change: {percent_change:.2f}%, annualized: {annualized_growth_rate:.2f}%"
            )
            print("")
            break

In [ ]:
# get last values of price_oracle and price_scale
for pool in pools:
    p_o = pools_data[pool]["price_oracle"][-1]
    p_s = pools_data[pool]["price_scale"][-1]
    abs_diff = abs(p_o - p_s)
    rel_diff = abs_diff / p_o
    print(
        f"\n{pool}: \nprice_oracle: {p_o:4.1f}, \nprice_scale: {p_s:4.1f}, \nabs_diff: {abs_diff:4.1f}, \nrel_diff: {100*rel_diff:4.2f}%"
    )

In [ ]:
# find where xcp_profit_half is closest to virtual_price after ts_fix
ts_fix = 1758733775
# and get latest timestamp of three
# init zero Timestamp
ts_last = pd.to_datetime(0, unit="s")
for pool in pools:
    idx_start = np.where(pools_data[pool]["ts"] > ts_fix)[0][0]
    xcp_profit_half_after_fix = pools_data[pool]["xcp_profit_half"][idx_start:]
    virtual_price_after_fix = pools_data[pool]["virtual_price"][idx_start:]
    diff = np.abs(xcp_profit_half_after_fix - virtual_price_after_fix)
    # first diff less than 0.001
    idx_min = np.where(diff < 0.002)[0][0]
    # print(np.where(diff<1))
    # print(idx_min)
    # print(f"diff: {diff[idx_min]}")
    ts_min = pools_data[pool]["ts_dt"][idx_start + idx_min]
    print(f"{pool}: xcp_profit_half meets virtual_price at {ts_min} (ts: {ts_min.timestamp():.0f})")
    if ts_min > ts_last:
        ts_last = ts_min
        pool_last = pool
print(f"Latest timestamp: {ts_last} for pool {pool_last}, (ts: {ts_last.timestamp()})")

In [ ]:
for pool in pools:
    spot_price_in = pools_data[pool]["spot_price_in"]
    spot_price_out = pools_data[pool]["spot_price_out"]
    diff = np.abs(spot_price_in - spot_price_out)
    rel_diff = diff / spot_price_in
    print(
        f"{pool}: spot_price_in: {spot_price_in[-1]:4.1f}, spot_price_out: {spot_price_out[-1]:4.1f}, diff: {diff[-1]:4.1f}, rel_diff: {100*rel_diff[-1]:4.2f}%"
    )